In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import pingouin as pg
import sys

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Matplotlib": plt.matplotlib.__version__,
    "Seaborn": sns.__version__,
    "Statsmodels": sm.__version__,
    "Pingouin": pg.__version__
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
df_versions

,Library,Version
0,Python,3.13.9
1,Pandas,2.3.3
2,NumPy,2.3.4
3,Matplotlib,3.10.7
4,Seaborn,0.13.2
5,Statsmodels,0.14.5
6,Pingouin,0.5.5


## Read Data

In [2]:
# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# reset options
# pd.reset_option('display.max_columns')

In [3]:
# load using pyarrow for performance (crime data is joined between crime & neighborhood & police using geom)
df_crime = pd.read_csv("../Data/chicago_crimes_export.csv", engine="pyarrow", dtype_backend="pyarrow")
df_arrest = pd.read_csv("../Data/chicago_arrests_export.csv", engine="pyarrow", dtype_backend="pyarrow")

## Check Data

In [4]:
df_crime.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
case_number,8353122,8352516,HZ140230,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8353122,NaN,NaN,NaN,2011-07-31 14:09:03,2001-01-01 00:00:00,2005-06-14 21:30:00,2010-06-05 04:42:00,2017-05-08 09:19:15,2025-12-24 00:00:00,NaN
block,8353122,63692,001XX N STATE ST,16996,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iucr,8353122,416,0820,671522,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_type,8353122,34,THEFT,1769511,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,8353122,565,SIMPLE,988659,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_description,8343134,218,STREET,2194409,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrest,8353122,2,f,6247440,NaN,NaN,NaN,NaN,NaN,NaN,NaN
domestic,8353122,2,f,6905712,NaN,NaN,NaN,NaN,NaN,NaN,NaN
beat,8353122.0,<NA>,<NA>,<NA>,1181.414643,111.0,621.0,1034.0,1731.0,2535.0,703.134414


In [5]:
df_crime.shape

(8353122, 23)

In [6]:
print(f'Police District (Police): {df_crime.p_district.nunique()}')
print(f'Police Sector (Police): {df_crime.p_sector.nunique()}')
print(f'Beat (Police): {df_crime.p_beat.nunique()}')

Police District (Police): 22
Police Sector (Police): 4
Beat (Police): 274


In [7]:
print(f'Police beats (Crime): {df_crime.beat.nunique()}')
print(f'Police District(Crime): {df_crime.district.nunique()}')
print(f'Ward (Crime): {df_crime.ward.nunique()}')

Police beats (Crime): 304
Police District(Crime): 24
Ward (Crime): 50


In [8]:
print(f"Beat Difference: {(df_crime.beat != df_crime.p_beat).sum():,}")
print(f"District Difference: {(df_crime.district != df_crime.p_district).sum():,}")

Beat Difference: 1,572,683
District Difference: 124,361


## Data Wrangle

In [9]:
# num of rows before 
before = df_crime.shape[0]
# remove any duplicates
df_crime = df_crime.drop_duplicates().reset_index(drop=True)
# num of rows after 
after = df_crime.shape[0]
print("Duplicate Rows Removed:", before - after)
print("Shape:", df_crime.shape)

Duplicate Rows Removed: 179
Shape: (8352943, 23)


In [10]:
# display number of unique values
for i in df_crime.columns:
    print(f"{i}: {df_crime[i].nunique():,}")

case_number: 8,352,516
date: 3,518,287
block: 63,692
iucr: 416
primary_type: 34
description: 565
location_description: 218
arrest: 2
domestic: 2
beat: 304
district: 24
ward: 50
community_area: 78
year: 25
updated_on: 5,908
zip_code: 59
zip_code_area: 59
primary_neighborhood: 98
secondary_neighborhood: 78
neighborhood_area: 98
p_district: 22
p_sector: 4
p_beat: 274


In [11]:
# aggregate data
aggregate = df_crime.groupby(['year','primary_type', 'primary_neighborhood']).size().unstack().fillna(0)
aggregate.head()

primary_neighborhood      Albany Park  Andersonville  Archer Heights  \
year primary_type                                                      
2001 ARSON                       12.0            1.0             3.0   
     ASSAULT                    232.0           35.0            76.0   
     BATTERY                    802.0           76.0           222.0   
     BURGLARY                   245.0           59.0           119.0   
     CRIM SEXUAL ASSAULT         18.0            0.0             4.0   

primary_neighborhood      Armour Square  Ashburn  Auburn Gresham  Austin  \
year primary_type                                                          
2001 ARSON                          4.0      8.0            14.0    72.0   
     ASSAULT                       55.0    276.0           879.0  1695.0   
     BATTERY                      210.0    674.0          2796.0  5594.0   
     BURGLARY                      58.0    249.0           811.0   936.0   
     CRIM SEXUAL ASSAULT            3.0     12.0            51.0   136.0   

primary_neighborhood      Avalon Park  Avondale  Belmont Cragin  Beverly  \
year primary_type                                                          
2001 ARSON                        3.0      18.0            27.0      1.0   
     ASSAULT                    167.0     280.0           565.0    116.0   
     BATTERY                    430.0     754.0          1379.0    212.0   
     BURGLARY                   122.0     420.0           669.0    138.0   
     CRIM SEXUAL ASSAULT          5.0      14.0            19.0      3.0   

primary_neighborhood      Boystown  Bridgeport  Brighton Park  Bucktown  \
year primary_type                                                         
2001 ARSON                     1.0        20.0           15.0       4.0   
     ASSAULT                  23.0       185.0          258.0     112.0   
     BATTERY                  71.0       546.0          757.0     216.0   
     BURGLARY                 26.0       331.0          397.0     275.0   
     CRIM SEXUAL ASSAULT       0.0         5.0           10.0       9.0   

primary_neighborhood      Burnside  Calumet Heights  Chatham  Chicago Lawn  \
year primary_type                                                            
2001 ARSON                     7.0              6.0     18.0          29.0   
     ASSAULT                  59.0            197.0    567.0         667.0   
     BATTERY                 165.0            426.0   1585.0        2155.0   
     BURGLARY                 26.0            161.0    529.0         826.0   
     CRIM SEXUAL ASSAULT       7.0              9.0     38.0          50.0   

primary_neighborhood      Chinatown  Clearing  Douglas  Dunning  East Side  \
year primary_type                                                            
2001 ARSON                      0.0       9.0      6.0      4.0        6.0   
     ASSAULT                   32.0     106.0    509.0    167.0      182.0   
     BATTERY                   67.0     260.0   1765.0    419.0      466.0   
     BURGLARY                  33.0     154.0    177.0    204.0      145.0   
     CRIM SEXUAL ASSAULT        1.0       5.0     35.0      8.0        4.0   

primary_neighborhood      East Village  Edgewater  Edison Park  Englewood  \
year primary_type                                                           
2001 ARSON                         4.0       10.0          1.0       73.0   
     ASSAULT                      63.0      304.0         25.0     2159.0   
     BATTERY                     162.0      865.0         66.0     7130.0   
     BURGLARY                    132.0      284.0         33.0     1380.0   
     CRIM SEXUAL ASSAULT           1.0       22.0          1.0      134.0   

primary_neighborhood      Fuller Park  Gage Park  Galewood  Garfield Park  \
year primary_type                                                           
2001 ARSON                        2.0       12.0       1.0           29.0   
     ASSAULT                    106.0 

In [12]:
list(aggregate.index)

[(2001, 'ARSON'),
 (2001, 'ASSAULT'),
 (2001, 'BATTERY'),
 (2001, 'BURGLARY'),
 (2001, 'CRIM SEXUAL ASSAULT'),
 (2001, 'CRIMINAL DAMAGE'),
 (2001, 'CRIMINAL SEXUAL ASSAULT'),
 (2001, 'CRIMINAL TRESPASS'),
 (2001, 'DECEPTIVE PRACTICE'),
 (2001, 'DOMESTIC VIOLENCE'),
 (2001, 'GAMBLING'),
 (2001, 'HOMICIDE'),
 (2001, 'INTERFERENCE WITH PUBLIC OFFICER'),
 (2001, 'INTIMIDATION'),
 (2001, 'KIDNAPPING'),
 (2001, 'LIQUOR LAW VIOLATION'),
 (2001, 'MOTOR VEHICLE THEFT'),
 (2001, 'NARCOTICS'),
 (2001, 'OBSCENITY'),
 (2001, 'OFFENSE INVOLVING CHILDREN'),
 (2001, 'OTHER NARCOTIC VIOLATION'),
 (2001, 'OTHER OFFENSE'),
 (2001, 'PROSTITUTION'),
 (2001, 'PUBLIC INDECENCY'),
 (2001, 'PUBLIC PEACE VIOLATION'),
 (2001, 'RITUALISM'),
 (2001, 'ROBBERY'),
 (2001, 'SEX OFFENSE'),
 (2001, 'STALKING'),
 (2001, 'THEFT'),
 (2001, 'WEAPONS VIOLATION'),
 (2002, 'ARSON'),
 (2002, 'ASSAULT'),
 (2002, 'BATTERY'),
 (2002, 'BURGLARY'),
 (2002, 'CRIM SEXUAL ASSAULT'),
 (2002, 'CRIMINAL DAMAGE'),
 (2002, 'CRIMINAL SEXUAL 

In [13]:
aggregate.loc[2025]

primary_neighborhood,Albany Park,Andersonville,Archer Heights,Armour Square,Ashburn,Auburn Gresham,Austin,Avalon Park,Avondale,Belmont Cragin,Beverly,Boystown,Bridgeport,Brighton Park,Bucktown,Burnside,Calumet Heights,Chatham,Chicago Lawn,Chinatown,Clearing,Douglas,Dunning,East Side,East Village,Edgewater,Edison Park,Englewood,Fuller Park,Gage Park,Galewood,Garfield Park,Garfield Ridge,Gold Coast,Grand Boulevard,Grand Crossing,Grant Park,Greektown,Hegewisch,Hermosa,Humboldt Park,Hyde Park,Irving Park,Jackson Park,Jefferson Park,Kenwood,Lake View,Lincoln Park,Lincoln Square,"Little Italy, UIC",Little Village,Logan Square,Loop,Lower West Side,Magnificent Mile,Mckinley Park,Millenium Park,Montclare,Morgan Park,Mount Greenwood,Museum Campus,Near South Side,New City,North Center,North Lawndale,North Park,Norwood Park,O'Hare,Oakland,Old Town,Portage Park,Printers Row,Pullman,River North,Riverdale,Rogers Park,Roseland,Rush & Division,"Sauganash,Forest Glen",Sheffield & DePaul,South Chicago,South Deering,South Shore,Streeterville,Ukrainian Village,United Center,Uptown,Washington Heights,Washington Park,West Elsdon,West Lawn,West Loop,West Pullman,West Ridge,West Town,Wicker Park,Woodlawn,Wrigleyville
primary_type,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ARSON,2.0,0.0,0.0,0.0,4.0,14.0,30.0,3.0,3.0,2.0,1.0,0.0,2.0,3.0,0.0,1.0,3.0,9.0,15.0,0.0,2.0,1.0,4.0,2.0,0.0,4.0,1.0,20.0,2.0,5.0,1.0,8.0,5.0,0.0,2.0,16.0,1.0,0.0,1.0,6.0,11.0,0.0,2.0,0.0,0.0,5.0,1.0,0.0,2.0,2.0,6.0,5.0,11.0,4.0,0.0,3.0,0.0,1.0,1.0,0.0,1.0,0.0,12.0,1.0,11.0,0.0,2.0,0.0,0.0,1.0,4.0,1.0,1.0,3.0,5.0,6.0,8.0,0.0,0.0,1.0,12.0,3.0,14.0,0.0,0.0,2.0,5.0,1.0,4.0,5.0,2.0,6.0,15.0,2.0,3.0,3.0,5.0,0.0
ASSAULT,203.0,26.0,72.0,71.0,205.0,674.0,1102.0,110.0,135.0,357.0,73.0,24.0,184.0,189.0,38.0,28.0,142.0,554.0,537.0,36.0,110.0,281.0,111.0,158.0,22.0,229.0,20.0,1065.0,57.0,194.0,34.0,747.0,155.0,24.0,326.0,739.0,14.0,29.0,55.0,101.0,572.0,148.0,179.0,22.0,81.0,188.0,243.0,153.0,144.0,323.0,402.0,264.0,414.0,261.0,23.0,75.0,13.0,50.0,159.0,42.0,10.0,179.0,420.0,79.0,622.0,68.0,90.0,73.0,70.0,128.0,221.0,20.0,120.0,294.0,166.0,292.0,542.0,76.0,22.0,32.0,388.0,195.0,930.0,107.0,43.0,256.0,381.0,268.0,301.0,75.0,128.0,271.0,333.0,247.0,131.0,112.0,388.0,37.0
BATTERY,405.0,39.0,137.0,105.0,373.0,1231.0,2490.0,185.0,294.0,743.0,142.0,74.0,245.0,408.0,79.0,60.0,242.0,983.0,1003.0,70.0,200.0,485.0,245.0,256.0,33.0,438.0,36.0,2115.0,107.0,399.0,55.0,1540.0,291.0,80.0,558.0,1307.0,43.0,27.0,116.0,197.0,1335.0,259.0,377.0,60.0,164.0,328.0,539.0,224.0,285.0,514.0,838.0,445.0,996.0,528.0,74.0,139.0,27.0,107.0,344.0,89.0,38.0,348.0,778.0,126.0,1371.0,151.0,154.0,212.0,138.0,236.0,460.0,32.0,182.0,703.0,273.0,723.0,1036.0,193.0,49.0,68.0,747.0,329.0,1759.0,277.0,90.0,404.0,768.0,443.0,565.0,170.0,252.0,519.0,588.0,575.0,265.0,209.0,748.0,118.0
BURGLARY,105.0,40.0,42.0,33.0,70.0,235.0,331.0,55.0,128.0,185.0,43.0,15.0,85.0,91.0,94.0,11.0,54.0,225.0,137.0,14.0,46.0,91.0,88.0,39.0,30.0,128.0,28.0,260.0,16.0,74.0,25.0,169.0,71.0,7.0,112.0,200.0,13.0,7.0,25.0,36.0,218.0,99.0,164.0,17.0,47.0,75.0,270.0,206.0,95.0,130.0,108.0,199.0,157.0,112.0,7.0,54.0,1.0,45.0,53.0,20.0,1.0,97.0,144.0,100.0,110.0,64.0,100.0,18.0,18.0,63.0,168.0,14.0,22.0,186.0,20.0,133.0,172.0,10.0,50.0,33.0,157.0,54.0,349.0,33.0,24.0,75.0,204.0,107.0,57.0,35.0,64.0,271.0,165.0,198.0,305.0,119.0,134.0,27.0
CONCEALED CARRY LICENSE VIOLATION,0.0,0.0,1.0,0.0,1.0,3.0,7.0,1.0,1.0,6.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0,4.0,1.0,1.0,3.0,1.0,0.0,2.0,1.0,0.0,7.0,0.0,0.0,1.0,4.0,24.0,1.0,2.0,9.0,1.0,1.0,0.0,1.0,4.0,1.0,1.0,2.0,0.0,0.0,7.0,0.0,0.0,9.0,7.0,4.0,11.0,5.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,4.0,4.0,0.0,19.0,0.0,0.0,55.0,0.0,1.0,2.0,0.0,0.0,3.0,0.0,1.0,4.0,0.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,4.0,0.0,3.0,1.0,1.0,3.0,3.0,3.0,1.0,3.0,0.0,4.0,0.0
CRIMINAL DAMAGE,295.0,42.0,83.0,94.0,232.0,862.0,1145.0,153.0,216.0,410.0,110.0,30.0,214.0,246.0,179.0,35.0,213.0,615.0,525.0,49.0,137.0,334.0,162.0,215.0,4

In [14]:
aggregate.loc[2025].loc[['ROBBERY', 'HOMICIDE']]

primary_neighborhood,Albany Park,Andersonville,Archer Heights,Armour Square,Ashburn,Auburn Gresham,Austin,Avalon Park,Avondale,Belmont Cragin,Beverly,Boystown,Bridgeport,Brighton Park,Bucktown,Burnside,Calumet Heights,Chatham,Chicago Lawn,Chinatown,Clearing,Douglas,Dunning,East Side,East Village,Edgewater,Edison Park,Englewood,Fuller Park,Gage Park,Galewood,Garfield Park,Garfield Ridge,Gold Coast,Grand Boulevard,Grand Crossing,Grant Park,Greektown,Hegewisch,Hermosa,Humboldt Park,Hyde Park,Irving Park,Jackson Park,Jefferson Park,Kenwood,Lake View,Lincoln Park,Lincoln Square,"Little Italy, UIC",Little Village,Logan Square,Loop,Lower West Side,Magnificent Mile,Mckinley Park,Millenium Park,Montclare,Morgan Park,Mount Greenwood,Museum Campus,Near South Side,New City,North Center,North Lawndale,North Park,Norwood Park,O'Hare,Oakland,Old Town,Portage Park,Printers Row,Pullman,River North,Riverdale,Rogers Park,Roseland,Rush & Division,"Sauganash,Forest Glen",Sheffield & DePaul,South Chicago,South Deering,South Shore,Streeterville,Ukrainian Village,United Center,Uptown,Washington Heights,Washington Park,West Elsdon,West Lawn,West Loop,West Pullman,West Ridge,West Town,Wicker Park,Woodlawn,Wrigleyville
primary_type,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ROBBERY,48.0,2.0,23.0,33.0,34.0,162.0,338.0,29.0,39.0,89.0,14.0,15.0,26.0,37.0,30.0,10.0,25.0,162.0,122.0,34.0,11.0,57.0,14.0,19.0,10.0,57.0,0.0,287.0,38.0,56.0,1.0,269.0,34.0,12.0,66.0,210.0,8.0,4.0,13.0,27.0,191.0,48.0,36.0,9.0,16.0,32.0,112.0,46.0,28.0,55.0,127.0,79.0,218.0,63.0,11.0,19.0,6.0,6.0,37.0,6.0,4.0,36.0,103.0,17.0,174.0,18.0,9.0,2.0,12.0,27.0,39.0,8.0,9.0,152.0,32.0,82.0,157.0,33.0,3.0,9.0,105.0,57.0,222.0,17.0,21.0,63.0,81.0,63.0,38.0,23.0,31.0,95.0,68.0,66.0,77.0,50.0,76.0,26.0
HOMICIDE,2.0,1.0,2.0,2.0,2.0,18.0,44.0,2.0,3.0,3.0,1.0,0.0,3.0,3.0,1.0,1.0,4.0,8.0,6.0,0.0,1.0,4.0,0.0,6.0,0.0,2.0,0.0,18.0,3.0,3.0,1.0,24.0,1.0,0.0,3.0,22.0,0.0,0.0,2.0,0.0,17.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,7.0,13.0,1.0,5.0,2.0,0.0,1.0,0.0,0.0,6.0,0.0,0.0,4.0,6.0,1.0,21.0,0.0,1.0,2.0,0.0,1.0,3.0,0.0,0.0,9.0,8.0,3.0,18.0,0.0,0.0,1.0,9.0,5.0,21.0,2.0,0.0,6.0,2.0,3.0,5.0,0.0,2.0,6.0,6.0,3.0,1.0,2.0,6.0,0.0


In [15]:
aggregate.loc[
    [(2024, 'ROBBERY'), (2024, 'HOMICIDE'),
     (2025, 'ROBBERY'), (2025, 'HOMICIDE')]
]


primary_neighborhood  Albany Park  Andersonville  Archer Heights  \
year primary_type                                                  
2024 ROBBERY                 72.0            2.0            38.0   
     HOMICIDE                 5.0            0.0             0.0   
2025 ROBBERY                 48.0            2.0            23.0   
     HOMICIDE                 2.0            1.0             2.0   

primary_neighborhood  Armour Square  Ashburn  Auburn Gresham  Austin  \
year primary_type                                                      
2024 ROBBERY                   38.0     49.0           231.0   610.0   
     HOMICIDE                   1.0      5.0            30.0    48.0   
2025 ROBBERY                   33.0     34.0           162.0   338.0   
     HOMICIDE                   2.0      2.0            18.0    44.0   

primary_neighborhood  Avalon Park  Avondale  Belmont Cragin  Beverly  \
year primary_type                                                      
2024 ROBBERY                 32.0      74.0           165.0     18.0   
     HOMICIDE                 2.0       0.0             4.0      1.0   
2025 ROBBERY                 29.0      39.0            89.0     14.0   
     HOMICIDE                 2.0       3.0             3.0      1.0   

primary_neighborhood  Boystown  Bridgeport  Brighton Park  Bucktown  Burnside  \
year primary_type                                                               
2024 ROBBERY              12.0        47.0           82.0      39.0      15.0   
     HOMICIDE              0.0         3.0            5.0       0.0       0.0   
2025 ROBBERY              15.0        26.0           37.0      30.0      10.0   
     HOMICIDE              0.0         3.0            3.0       1.0       1.0   

primary_neighborhood  Calumet Heights  Chatham  Chicago Lawn  Chinatown  \
year primary_type                                                         
2024 ROBBERY                     50.0    266.0         193.0       56.0   
     HOMICIDE                     5.0     25.0           8.0        4.0   
2025 ROBBERY                     25.0    162.0         122.0       34.0   
     HOMICIDE                     4.0      8.0           6.0        0.0   

primary_neighborhood  Clearing  Douglas  Dunning  East Side  East Village  \
year primary_type                                                           
2024 ROBBERY              19.0     76.0     20.0       22.0          25.0   
     HOMICIDE              0.0      9.0      1.0        3.0           0.0   
2025 ROBBERY              11.0     57.0     14.0       19.0          10.0   
     HOMICIDE              1.0      4.0      0.0        6.0           0.0   

primary_neighborhood  Edgewater  Edison Park  Englewood  Fuller Park  \
year primary_type                                                      
2024 ROBBERY               93.0          3.0      447.0         44.0   
     HOMICIDE               2.0          0.0       38.0          3.0   
2025 ROBBERY               57.0          0.0      287.0         38.0   
     HOMICIDE               2.0          0.0       18.0          3.0   

primary_neighborhood  Gage Park  Galewood  Garfield Park  Garfield Ridge  \
year primary_type                                                          
2024 ROBBERY              128.0       8.0          363.0            44.0   
     HOMICIDE              12.0       0.0           34.0             3.0   
2025 ROBBERY               56.0       1.0          269.0            34.0   
     HOMICIDE               3.0       1.0           24.0             1.0   

primary_neighborhood  Gold Coast  Grand Boulevard  Grand Crossing  Grant Park  \
year primary_type                                                               
2024 ROBBERY                23.0             83.0           272.0        12.0   
     HOMICIDE                1.0              7.0            34.0         0.0   
2025 ROBBERY                12.0             66.0           210.0         8.0   
     HOMICIDE                

In [16]:
aggregate.loc[(slice(2024, 2025), ['ROBBERY', 'HOMICIDE']), :]

primary_neighborhood  Albany Park  Andersonville  Archer Heights  \
year primary_type                                                  
2024 HOMICIDE                 5.0            0.0             0.0   
     ROBBERY                 72.0            2.0            38.0   
2025 HOMICIDE                 2.0            1.0             2.0   
     ROBBERY                 48.0            2.0            23.0   

primary_neighborhood  Armour Square  Ashburn  Auburn Gresham  Austin  \
year primary_type                                                      
2024 HOMICIDE                   1.0      5.0            30.0    48.0   
     ROBBERY                   38.0     49.0           231.0   610.0   
2025 HOMICIDE                   2.0      2.0            18.0    44.0   
     ROBBERY                   33.0     34.0           162.0   338.0   

primary_neighborhood  Avalon Park  Avondale  Belmont Cragin  Beverly  \
year primary_type                                                      
2024 HOMICIDE                 2.0       0.0             4.0      1.0   
     ROBBERY                 32.0      74.0           165.0     18.0   
2025 HOMICIDE                 2.0       3.0             3.0      1.0   
     ROBBERY                 29.0      39.0            89.0     14.0   

primary_neighborhood  Boystown  Bridgeport  Brighton Park  Bucktown  Burnside  \
year primary_type                                                               
2024 HOMICIDE              0.0         3.0            5.0       0.0       0.0   
     ROBBERY              12.0        47.0           82.0      39.0      15.0   
2025 HOMICIDE              0.0         3.0            3.0       1.0       1.0   
     ROBBERY              15.0        26.0           37.0      30.0      10.0   

primary_neighborhood  Calumet Heights  Chatham  Chicago Lawn  Chinatown  \
year primary_type                                                         
2024 HOMICIDE                     5.0     25.0           8.0        4.0   
     ROBBERY                     50.0    266.0         193.0       56.0   
2025 HOMICIDE                     4.0      8.0           6.0        0.0   
     ROBBERY                     25.0    162.0         122.0       34.0   

primary_neighborhood  Clearing  Douglas  Dunning  East Side  East Village  \
year primary_type                                                           
2024 HOMICIDE              0.0      9.0      1.0        3.0           0.0   
     ROBBERY              19.0     76.0     20.0       22.0          25.0   
2025 HOMICIDE              1.0      4.0      0.0        6.0           0.0   
     ROBBERY              11.0     57.0     14.0       19.0          10.0   

primary_neighborhood  Edgewater  Edison Park  Englewood  Fuller Park  \
year primary_type                                                      
2024 HOMICIDE               2.0          0.0       38.0          3.0   
     ROBBERY               93.0          3.0      447.0         44.0   
2025 HOMICIDE               2.0          0.0       18.0          3.0   
     ROBBERY               57.0          0.0      287.0         38.0   

primary_neighborhood  Gage Park  Galewood  Garfield Park  Garfield Ridge  \
year primary_type                                                          
2024 HOMICIDE              12.0       0.0           34.0             3.0   
     ROBBERY              128.0       8.0          363.0            44.0   
2025 HOMICIDE               3.0       1.0           24.0             1.0   
     ROBBERY               56.0       1.0          269.0            34.0   

primary_neighborhood  Gold Coast  Grand Boulevard  Grand Crossing  Grant Park  \
year primary_type                                                               
2024 HOMICIDE                1.0              7.0            34.0         0.0   
     ROBBERY                23.0             83.0           272.0        12.0   
2025 HOMICIDE                0.0              3.0            22.0         0.0   
     ROBBERY                1

In [17]:
reset_agg = aggregate.reset_index()
reset_agg.head()

primary_neighborhood,year,primary_type,Albany Park,Andersonville,Archer Heights,Armour Square,Ashburn,Auburn Gresham,Austin,Avalon Park,Avondale,Belmont Cragin,Beverly,Boystown,Bridgeport,Brighton Park,Bucktown,Burnside,Calumet Heights,Chatham,Chicago Lawn,Chinatown,Clearing,Douglas,Dunning,East Side,East Village,Edgewater,Edison Park,Englewood,Fuller Park,Gage Park,Galewood,Garfield Park,Garfield Ridge,Gold Coast,Grand Boulevard,Grand Crossing,Grant Park,Greektown,Hegewisch,Hermosa,Humboldt Park,Hyde Park,Irving Park,Jackson Park,Jefferson Park,Kenwood,Lake View,Lincoln Park,Lincoln Square,"Little Italy, UIC",Little Village,Logan Square,Loop,Lower West Side,Magnificent Mile,Mckinley Park,Millenium Park,Montclare,Morgan Park,Mount Greenwood,Museum Campus,Near South Side,New City,North Center,North Lawndale,North Park,Norwood Park,O'Hare,Oakland,Old Town,Portage Park,Printers Row,Pullman,River North,Riverdale,Rogers Park,Roseland,Rush & Division,"Sauganash,Forest Glen",Sheffield & DePaul,South Chicago,South Deering,South Shore,Streeterville,Ukrainian Village,United Center,Uptown,Washington Heights,Washington Park,West Elsdon,West Lawn,West Loop,West Pullman,West Ridge,West Town,Wicker Park,Woodlawn,Wrigleyville
0,2001,ARSON,12.0,1.0,3.0,4.0,8.0,14.0,72.0,3.0,18.0,27.0,1.0,1.0,20.0,15.0,4.0,7.0,6.0,18.0,29.0,0.0,9.0,6.0,4.0,6.0,4.0,10.0,1.0,73.0,2.0,12.0,1.0,29.0,7.0,0.0,10.0,22.0,0.0,0.0,4.0,23.0,43.0,3.0,17.0,0.0,2.0,4.0,7.0,1.0,5.0,8.0,37.0,24.0,0.0,23.0,0.0,7.0,0.0,5.0,7.0,3.0,0.0,3.0,40.0,9.0,30.0,1.0,1.0,1.0,2.0,1.0,11.0,0.0,2.0,7.0,4.0,13.0,37.0,0.0,1.0,3.0,14.0,9.0,15.0,0.0,2.0,2.0,13.0,8.0,12.0,5.0,11.0,1.0,17.0,8.0,14.0,7.0,18.0,0.0
1,2001,ASSAULT,232.0,35.0,76.0,55.0,276.0,879.0,1695.0,167.0,280.0,565.0,116.0,23.0,185.0,258.0,112.0,59.0,197.0,567.0,667.0,32.0,106.0,509.0,167.0,182.0,63.0,304.0,25.0,2159.0,106.0,235.0,67.0,892.0,265.0,29.0,702.0,855.0,4.0,20.0,64.0,220.0,1375.0,170.0,273.0,25.0,124.0,207.0,308.0,196.0,239.0,325.0,521.0,706.0,377.0,456.0,49.0,133.0,2.0,87.0,269.0,81.0,7.0,200.0,735.0,211.0,844.0,76.0,115.0,117.0,144.0,146.0,326.0,18.0,173.0,408.0,262.0,526.0,1035.0,77.0,35.0,37.0,720.0,216.0,1158.0,70.0,79.0,318.0,546.0,390.0,428.0,71.0,156.0,155.0,624.0,274.0,259.0,273.0,633.0,21.0
2,2001,BATTERY,802.0,76.0,222.0,210.0,674.0,2796.0,5594.0,430.0,754.0,1379.0,212.0,71.0,546.0,757.0,216.0,165.0,426.0,1585.0,2155.0,67.0,260.0,1765.0,419.0,466.0,162.0,865.0,66.0,7130.0,333.0,668.0,153.0,3127.0,619.0,158.0,2667.0,2718.0,31.0,58.0,195.0,592.0,3977.0,424.0,990.0,64.0,265.0,554.0,845.0,498.0,622.0,998.0,1715.0,1790.0,684.0,1095.0,76.0,304.0,5.0,196.0,653.0,170.0,39.0,686.0,2396.0,501.0,2971.0,182.0,284.0,291.0,392.0,352.0,842.0,47.0,380.0,1644.0,888.0,1647.0,2922.0,261.0,85.0,118.0,2009.0,596.0,3354.0,187.0,235.0,1095.0,1637.0,934.0,1580.0,199.0,450.0,308.0,1753.0,776.0,731.0,717.0,1923.0,115.0
3,2001,BURGLARY,245.0,59.0,119.0,58.0,249.0,811.0,936.0,122.0,420.0,669.0,138.0,26.0,331.0,397.0,275.0,26.0,161.0,529.0,826.0,33.0,154.0,177.0,204.0,145.0,132.0,284.0,33.0,1380.0,70.0,342.0,111.0,517.0,293.0,23.0,364.0,568.0,3.0,7.0,54.0,218.0,861.0,165.0,389.0,29.0,131.0,132.0,612.0,369.0,264.0,204.0,580.0,672.0,202.0,263.0,26.0,217.0,1.0,85.0,213.0,56.0,4.0,71.0,619.0,332.0,437.0,93.0,138.0,44.0,50.0,128.0,455.0,14.0,80.0,299.0,106.0,382.0,606.0,37.0,79.0,92.0,359.0,136.0,776.0,61.0,165.0,131.0,261.0,257.0,212.0,156.0,207.0,146.0,485.0,334.0,395.0,440.0,329.0,47.0
4,2001,CRIM SEXUAL ASSAULT,18.0,0.0,4.0,3.0,12.0,51.0,136.0,5.0,14.0,19.0,3.0,0.0,5.0,10.0,9.0,7.0,9.0,38.0,50.0,1.0,5.0,35.0,8.0,4.0,1.0,22.0,1.0,134.0,3.0,9.0,4.0,60.0,12.0,2.0,45.0,47.0,0.0,1.0,4.0,16.0,54.0,13.0,22.0,4.0,6.0,11.0,15.0,11.0,9.0,20.0,28.0,33.0,12.0,12.0,0.0,4.0,0.0,2.0,7.0,1.0,1.0,11.0,44.0,6.0,50.0,5.0,2.0,3.0,4.0,9.0,10.0,0.0,10.0,20.0,13.0,26.0,56.0,5.0,4.0,6.0,43.0,11.0,66.0,3.0,4.0,13.0,28.0,16.0,40.0,4.0,5.0,10.0,35.0,25.0,13.0,26.0,39.0,1.0


## New Data Set

In [18]:
reset_agg.primary_type.unique()

<ArrowExtensionArray>
[                            'ARSON',                           'ASSAULT',
                           'BATTERY',                          'BURGLARY',
               'CRIM SEXUAL ASSAULT',                   'CRIMINAL DAMAGE',
           'CRIMINAL SEXUAL ASSAULT',                 'CRIMINAL TRESPASS',
                'DECEPTIVE PRACTICE',                 'DOMESTIC VIOLENCE',
                          'GAMBLING',                          'HOMICIDE',
  'INTERFERENCE WITH PUBLIC OFFICER',                      'INTIMIDATION',
                        'KIDNAPPING',              'LIQUOR LAW VIOLATION',
               'MOTOR VEHICLE THEFT',                         'NARCOTICS',
                         'OBSCENITY',        'OFFENSE INVOLVING CHILDREN',
          'OTHER NARCOTIC VIOLATION',                     'OTHER OFFENSE',
                      'PROSTITUTION',                  'PUBLIC INDECENCY',
            'PUBLIC PEACE VIOLATION',                         'RITUALISM',
   

In [19]:
# rearrange data
neighborhood_cols = reset_agg.columns[2:]   # all neighborhood columns

df_long = reset_agg.melt(
    id_vars=['year', 'primary_type'],
    value_vars=neighborhood_cols,
    var_name='neighborhood',
    value_name='crime_count'
)
df_long.head()

,year,primary_type,neighborhood,crime_count
0,2001,ARSON,Albany Park,12.0
1,2001,ASSAULT,Albany Park,232.0
2,2001,BATTERY,Albany Park,802.0
3,2001,BURGLARY,Albany Park,245.0
4,2001,CRIM SEXUAL ASSAULT,Albany Park,18.0


#### Calculating the Z-score highlights specialized anomalies—places where a specific crime type is occurring at a rate far beyond what is statistically normal for the rest of the city.
- A Z-score above 3.0 is typically considered a significant outlier; seeing scores of 5.0 to 8.0 indicates extreme geographic concentration.

In [20]:
# Calculate mean and standard deviation per year/type
stats = df_long.groupby(['year', 'primary_type'], observed=False)['crime_count'].agg(['mean', 'std']).fillna(0)

# Merge back and calculate Z-Score
df_long = df_long.merge(stats, on=['year', 'primary_type'])
df_long['z_score'] = (df_long['crime_count'] - df_long['mean']) / df_long['std']

# Find the most "Extreme" statistical outliers
extremes = df_long[df_long['year'] == 2025].sort_values('z_score', ascending=False)

In [21]:
extremes[['neighborhood', 'primary_type', 'crime_count', 'z_score']].head(10)

,neighborhood,primary_type,crime_count,z_score
51993,O'Hare,CONCEALED CARRY LICENSE VIOLATION,55.0,8.090589
25237,Garfield Ridge,PROSTITUTION,70.0,7.993028
40535,Loop,OTHER NARCOTIC VIOLATION,3.0,6.926511
5334,Austin,HOMICIDE,44.0,5.831036
40538,Loop,PUBLIC INDECENCY,2.0,5.641306
40532,Loop,NON-CRIMINAL,1.0,5.598530
58127,Rogers Park,NON-CRIMINAL,1.0,5.598530
31352,Humboldt Park,NON-CRIMINAL,1.0,5.598530
31351,Humboldt Park,NARCOTICS,1134.0,5.349758
40521,Loop,CRIMINAL TRESPASS,310.0,5.290609


In [22]:
extremes.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
year,3038.0,<NA>,<NA>,<NA>,2025.0,0.0,2025.0,2025.0,2025.0,2025.0,2025.0
primary_type,3038,31,CONCEALED CARRY LICENSE VIOLATION,98,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighborhood,3038,98,O'Hare,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
crime_count,3038.0,NaN,NaN,NaN,75.725477,196.943123,0.0,0.0,5.0,54.75,2797.0
mean,3038.0,NaN,NaN,NaN,75.725477,128.932724,0.030612,1.387755,10.612245,96.285714,544.744898
std,3038.0,NaN,NaN,NaN,78.936518,127.115714,0.17315,1.875686,13.949565,114.01786,501.627913
z_score,3038.0,NaN,NaN,NaN,0.0,0.995049,-1.182882,-0.593138,-0.303296,0.144527,8.090589


### Descriptive & EDA

In [ ]:
crime_year = 

In [26]:
# take a look at 2024 & 2025
df_2025 = df_long.iloc[:, :4][df_long.year ==2025].reset_index(drop=True)
df_2024 = df_long.iloc[:, :4][df_long.year ==2024].reset_index(drop=True)

In [27]:
df_2025.head()

,year,primary_type,neighborhood,crime_count
0,2025,ARSON,Albany Park,2.0
1,2025,ASSAULT,Albany Park,203.0
2,2025,BATTERY,Albany Park,405.0
3,2025,BURGLARY,Albany Park,105.0
4,2025,CONCEALED CARRY LICENSE VIOLATION,Albany Park,0.0


In [31]:
df_2025.year.unique(), df_2024.year.unique()

(<ArrowExtensionArray>
 [2025]
 Length: 1, dtype: int64[pyarrow],
 <ArrowExtensionArray>
 [2024]
 Length: 1, dtype: int64[pyarrow])

## 2025 ANOVA 

In [36]:
# Normality test on the 'crime_count' column
normality_results = pg.normality(df_2025['crime_count'])

print("--- Normality Test Results ---")
print(normality_results)

--- Normality Test Results ---
                    W          pval  normal
crime_count  0.417191  1.241935e-71   False


- The data is extremely non-normal, and test homoscedasticity.

In [61]:
# Testing if variance in 'crime_count' is equal across different primary_type & neighborhood
homogeneity_p_results = pg.homoscedasticity(data=df_2025, dv='crime_count', group='primary_type')
homogeneity_n_results = pg.homoscedasticity(data=df_2025, dv='crime_count', group='neighborhood')


print("\n--- Homogeneity of Variance Results ---\n")
print("Homogeneity result for Type of Crime:\n", homogeneity_p_results.to_string())
print("\nHomogeneity result for Neighborhood:\n", homogeneity_n_results.to_string())


--- Homogeneity of Variance Results ---

Homogeneity result for Type of Crime:
                 W           pval  equal_var
levene  48.977947  1.036113e-233      False

Homogeneity result for Neighborhood:
                W          pval  equal_var
levene  4.161421  7.892982e-36      False


In [64]:
# Use Welch's ANOVA instead of regular ANOVA
welch_res_2025_p = pg.welch_anova(data=df_2025, dv='crime_count', between='primary_type')
welch_res_2025_n = pg.welch_anova(data=df_2025, dv='crime_count', between='neighborhood')
print(welch_res_2025_p.to_string())
print("\n", welch_res_2025_n.to_string())

         Source  ddof1       ddof2          F          p-unc       np2
0  primary_type     30  1058.13242  61.907701  4.669137e-209  0.428593

          Source  ddof1       ddof2         F         p-unc      np2
0  neighborhood     97  999.592027  4.573427  9.077035e-36  0.12136


(3038, 4)

- This is partial eta‑squared (np2), an effect size.
    - 0.01 = small
    - 0.06 = medium
    - 0.14 = large
    - 0.43 = extremely large

- Interpretation:
    - About 43% of all variance in crime_count is explained by crime type.
    - That is a huge effect.

In [45]:
# Testing if variance in 'crime_count' is equal across different 'neighborhood'
homogeneity_results = pg.homoscedasticity(data=df_2025, dv='crime_count', group='neighborhood')

print("\n--- Homogeneity of Variance Results ---")
print(homogeneity_results)


--- Homogeneity of Variance Results ---
               W          pval  equal_var
levene  4.161421  7.892982e-36      False


In [44]:
# Use Welch's ANOVA instead of regular ANOVA
welch_res_2025 = pg.welch_anova(data=df_2025, dv='crime_count', between='neighborhood')
welch_res_2025

,Source,ddof1,ddof2,F,p-unc,np2
0,neighborhood,97,999.592027,4.573427,9.077035e-36,0.12136


#### Conclusion
- Comparing Homoscedasticity and  for primary_type, which represents crime types, and neighborhood for 2025 reported crimes  

In [25]:
# The "Robust" T-test approach:
robust_crime = pg.pairwise_tests(
    data=df_2024_2025, 
    dv='crime_count', 
    between='neighborhood', 
    parametric=True, 
    correction=True # This handles the unequal variance (Welch's)
)

In [26]:
robust_crime.info()
robust_crime.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4753 entries, 0 to 4752
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Contrast     4753 non-null   object 
 1   A            4753 non-null   object 
 2   B            4753 non-null   object 
 3   Paired       4753 non-null   bool   
 4   Parametric   4753 non-null   bool   
 5   T            4753 non-null   float64
 6   dof          4753 non-null   float64
 7   alternative  4753 non-null   object 
 8   p-unc        4753 non-null   float64
 9   BF10         4753 non-null   object 
 10  hedges       4753 non-null   float64
dtypes: bool(2), float64(4), object(5)
memory usage: 343.6+ KB


,Contrast,A,B,Paired,Parametric,T,dof,alternative,p-unc,BF10,hedges
0,neighborhood,Albany Park,Andersonville,False,True,3.589389,68.846894,two-sided,0.000616,54.39,0.640702
1,neighborhood,Albany Park,Archer Heights,False,True,2.093167,94.884359,two-sided,0.039002,1.367,0.373628
2,neighborhood,Albany Park,Armour Square,False,True,3.299155,67.791680,two-sided,0.001548,23.24,0.588896
3,neighborhood,Albany Park,Ashburn,False,True,0.280644,119.816446,two-sided,0.779468,0.199,0.050095
4,neighborhood,Albany Park,Auburn Gresham,False,True,-2.768333,79.753337,two-sided,0.007003,5.787,-0.494144


In [29]:
# The "Robust" T-test approach with :
robust_crime_bonf = pg.pairwise_tests(
    data=df_2024_2025, 
    dv='crime_count', 
    between='neighborhood', 
    parametric=True, 
    correction=True, # This handles the unequal variance (Welch's)
    padjust='bonf'   # Bonferroni correction
)
#  statistical method to prevent false positives (Type I errors) 
# when performing multiple hypothesis tests by adjusting the significance level, 
# making it harder to declare results significant

In [30]:
robust_crime_bonf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4753 entries, 0 to 4752
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Contrast     4753 non-null   object 
 1   A            4753 non-null   object 
 2   B            4753 non-null   object 
 3   Paired       4753 non-null   bool   
 4   Parametric   4753 non-null   bool   
 5   T            4753 non-null   float64
 6   dof          4753 non-null   float64
 7   alternative  4753 non-null   object 
 8   p-unc        4753 non-null   float64
 9   p-corr       4753 non-null   float64
 10  p-adjust     4753 non-null   object 
 11  BF10         4753 non-null   object 
 12  hedges       4753 non-null   float64
dtypes: bool(2), float64(5), object(6)
memory usage: 417.9+ KB


In [36]:

robust_crime_bonf.BF10 = robust_crime_bonf.BF10.astype('float')
robust_crime_bonf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4753 entries, 0 to 4752
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Contrast     4753 non-null   object 
 1   A            4753 non-null   object 
 2   B            4753 non-null   object 
 3   Paired       4753 non-null   bool   
 4   Parametric   4753 non-null   bool   
 5   T            4753 non-null   float64
 6   dof          4753 non-null   float64
 7   alternative  4753 non-null   object 
 8   p-unc        4753 non-null   float64
 9   p-corr       4753 non-null   float64
 10  p-adjust     4753 non-null   object 
 11  BF10         4753 non-null   object 
 12  hedges       4753 non-null   float64
dtypes: bool(2), float64(5), object(6)
memory usage: 417.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4753 entries, 0 to 4752
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Cont

In [27]:
robust_crime.info()
robust_crime.BF10 = robust_crime.BF10.astype('float')
robust_crime.sort_values(by='BF10', ascending=False).head(10)

NameError: name 'robust_crime' is not defined

,Contrast,A,B,Paired,Parametric,T,dof,alternative,p-unc,BF10,hedges
3119,neighborhood,Humboldt Park,Museum Campus,False,True,4.866078,61.101534,two-sided,0.000008,4539.429,0.868590
2210,neighborhood,Edison Park,Humboldt Park,False,True,-4.837865,61.180189,two-sided,0.000009,4071.489,-0.863554
3115,neighborhood,Humboldt Park,Millenium Park,False,True,4.802582,61.570375,two-sided,0.000010,3555.859,0.857256
2865,neighborhood,Grant Park,Humboldt Park,False,True,-4.802491,61.310188,two-sided,0.000010,3554.624,-0.857239
1374,neighborhood,Burnside,Humboldt Park,False,True,-4.791841,61.207266,two-sided,0.000011,3412.755,-0.855338
3102,neighborhood,Humboldt Park,Jackson Park,False,True,4.778456,61.293646,two-sided,0.000011,3242.751,0.852949
2925,neighborhood,Greektown,Humboldt Park,False,True,-4.777810,61.476421,two-sided,0.000011,3234.774,-0.852834
2570,neighborhood,Garfield Park,Museum Campus,False,True,4.731846,61.105649,two-sided,0.000014,2716.409,0.844629
135,neighborhood,Andersonville,Humboldt Park,False,True,-4.719979,61.917833,two-sided,0.000014,2597.169,-0.842511
2067,neighborhood,East Village,Humboldt Park,False,True,-4.711790,61.719743,two-sided,0.000014,2518.081,-0.841049


In [37]:
robust_crime_bonf.BF10 = robust_crime_bonf.BF10.astype('float')
robust_crime_bonf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4753 entries, 0 to 4752
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Contrast     4753 non-null   object 
 1   A            4753 non-null   object 
 2   B            4753 non-null   object 
 3   Paired       4753 non-null   bool   
 4   Parametric   4753 non-null   bool   
 5   T            4753 non-null   float64
 6   dof          4753 non-null   float64
 7   alternative  4753 non-null   object 
 8   p-unc        4753 non-null   float64
 9   p-corr       4753 non-null   float64
 10  p-adjust     4753 non-null   object 
 11  BF10         4753 non-null   float64
 12  hedges       4753 non-null   float64
dtypes: bool(2), float64(6), object(5)
memory usage: 417.9+ KB


In [38]:
robust_crime_bonf.sort_values(by='BF10', ascending=False).head(10)

,Contrast,A,B,Paired,Parametric,T,dof,alternative,p-unc,p-corr,p-adjust,BF10,hedges
3119,neighborhood,Humboldt Park,Museum Campus,False,True,4.866078,61.101534,two-sided,0.000008,0.039736,bonf,4539.429,0.868590
2210,neighborhood,Edison Park,Humboldt Park,False,True,-4.837865,61.180189,two-sided,0.000009,0.043940,bonf,4071.489,-0.863554
3115,neighborhood,Humboldt Park,Millenium Park,False,True,4.802582,61.570375,two-sided,0.000010,0.049456,bonf,3555.859,0.857256
2865,neighborhood,Grant Park,Humboldt Park,False,True,-4.802491,61.310188,two-sided,0.000010,0.049796,bonf,3554.624,-0.857239
1374,neighborhood,Burnside,Humboldt Park,False,True,-4.791841,61.207266,two-sided,0.000011,0.051889,bonf,3412.755,-0.855338
3102,neighborhood,Humboldt Park,Jackson Park,False,True,4.778456,61.293646,two-sided,0.000011,0.054348,bonf,3242.751,0.852949
2925,neighborhood,Greektown,Humboldt Park,False,True,-4.777810,61.476421,two-sided,0.000011,0.054230,bonf,3234.774,-0.852834
2570,neighborhood,Garfield Park,Museum Campus,False,True,4.731846,61.105649,two-sided,0.000014,0.064606,bonf,2716.409,0.844629
135,neighborhood,Andersonville,Humboldt Park,False,True,-4.719979,61.917833,two-sided,0.000014,0.066140,bonf,2597.169,-0.842511
2067,neighborhood,East Village,Humboldt Park,False,True,-4.711790,61.719743,two-sided,0.000014,0.068441,bonf,2518.081,-0.841049


In [31]:
# This tests Year, Neighborhood, primary_type
model = ols('crime_count ~ C(year) + C(neighborhood) + C(primary_type)', data=df_long).fit()
anova_results = anova_lm(model, typ=2)

In [32]:
anova_results

,sum_sq,df,F,PR(>F)
C(year),7.683053e+07,24.0,54.782289,1.993514e-260
C(neighborhood),9.980606e+08,97.0,176.077068,0.000000e+00
C(primary_type),2.427117e+09,33.0,1258.619424,0.000000e+00
Residual,4.371909e+09,74815.0,NaN,NaN


In [33]:
print(anova_results)

                       sum_sq       df            F         PR(>F)
C(year)          7.683053e+07     24.0    54.782289  1.993514e-260
C(neighborhood)  9.980606e+08     97.0   176.077068   0.000000e+00
C(primary_type)  2.427117e+09     33.0  1258.619424   0.000000e+00
Residual         4.371909e+09  74815.0          NaN            NaN


In [40]:
# largest crime
df_long_filtered['primary_type'].value_counts().nlargest(5).index

Index(['ARSON', 'ASSAULT', 'BATTERY', 'BURGLARY',
       'CONCEALED CARRY LICENSE VIOLATION'],
      dtype='string[pyarrow]', name='primary_type')

In [34]:
# largest crime
df_long_filtered['neighborhood'].value_counts().nlargest(5).index

Index(['Albany Park', 'Andersonville', 'Archer Heights', 'Armour Square',
       'Ashburn'],
      dtype='string[pyarrow]', name='neighborhood')

In [29]:
# Create a count table
counts = df_long_filtered.groupby(['neighborhood', 'primary_type']).size().unstack(fill_value=0)

# Find combinations with only 1 sample (Standard Deviation is impossible)
single_samples = (counts == 1).sum().sum()

# Find combinations with 0 samples (Mean/SD are NaN)
zero_samples = (counts == 0).sum().sum()

print(f"Groups with 1 sample: {single_samples}")
print(f"Groups with 0 samples: {zero_samples}")

Groups with 1 sample: 0
Groups with 0 samples: 0


In [33]:
# 1. Filter out pairs that couldn't be calculated (NaN)
# 2. Filter for significant results (p < 0.05)
significant_results = posthocs.dropna(subset=['p-unc'])
significant_results = significant_results[significant_results['p-unc'] < 0.05]

print(significant_results.head().to_string())

        Contrast neighborhood            A              B Paired Parametric         T   dof alternative     p-unc   BF10    hedges
0   neighborhood            -  Albany Park  Andersonville  False       True  2.413472  18.0   two-sided  0.026678  2.592  1.033732
10  neighborhood            -  Albany Park       Boystown  False       True  2.219374  18.0   two-sided  0.039549  1.986  0.950596
14  neighborhood            -  Albany Park       Burnside  False       True  2.337110  18.0   two-sided  0.031189   2.33  1.001025
18  neighborhood            -  Albany Park      Chinatown  False       True  2.226112  18.0   two-sided  0.039020  2.004  0.953482
23  neighborhood            -  Albany Park   East Village  False       True  2.429926  18.0   two-sided  0.025790  2.653  1.040779


In [44]:
import pathlib

# This gets the directory you are currently working in
current_dir = pathlib.Path.cwd()
# pathlib.Path.cwd()
# # To go two levels up, just like your original code intended:
# grandparent_dir = current_dir.parent.parent.absolute()
current_dir.parent.parent.absolute()

PosixPath('/Users/sir/Desktop/Project')

In [46]:
try:
    # Works in .py scripts
    root = pathlib.Path(__file__).parent.parent.absolute()
except NameError:
    # Works in Jupyter/Interactive shells
    root = pathlib.Path.cwd().parent.parent.absolute()

print(f"Project root is: {root}")

Project root is: /Users/sir/Desktop/Project


In [ ]:
# Fixed Two‑way ANOVA
aov_inter = pg.anova(
    data=df_long_filtered,
    dv='crime_count',
    between=['neighborhood', 'primary_type'],
    detailed=True
)

print(aov_inter)

In [23]:
# Two‑way ANOVA with interaction
aov_inter = pg.anova(
    data=df_long_filtered,
    dv='crime_count',
    between=['neighborhood', 'primary_type'],
    detailed=True
)

print(aov_inter.to_string())

                        Source            SS    DF            MS            F  p-unc       np2
0                 neighborhood  3.176246e+07    97  3.274481e+05   451.180134    0.0  0.935089
1                 primary_type  1.133591e+08    30  3.778637e+06  5206.462288    0.0  0.980921
2  neighborhood * primary_type  1.143131e+08  2910  3.928285e+04    54.126573    0.0  0.981077
3                     Residual  2.204856e+06  3038  7.257591e+02          NaN    NaN       NaN
